# 03 — Prepare YOLO Dataset (Phase 3 — 10-class SG Taxonomy)

Builds the YOLO training dataset from Grounding DINO auto-labels
(`data/silver/gdino_v1/labels/`) + human-reviewed corrections.

**Input:** `data/silver/gdino_v1/labels/cam{id}_{YYYYMMDD}_{HHMMSS}.txt`  
**Source images:** `data/raw/{YYYY-MM-DD}/{camera_id}/{HH-MM-SS}.jpg`  
**Output:**
```
yolo_dataset/
  images/train|val|test/*.jpg
  labels/train|val|test/*.txt   ← YOLO format: class cx cy w h
  data.yaml                     ← nc=10, SG taxonomy
```

**Split:** 70/15/15 stratified by camera so every camera appears in train/val/test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path

RAW_DIR    = '/content/drive/MyDrive/sg_smart_city/data/raw'
LABELS_DIR = '/content/drive/MyDrive/sg_smart_city/data/silver/gdino_v1/labels'
OUTPUT_DIR = '/content/drive/MyDrive/sg_smart_city/data/yolo_dataset'

# 10-class SG taxonomy — matches notebooks/10_label_sg_vehicles.ipynb
CLASS_NAMES = [
    'car', 'motorcycle', 'scooter', 'bus', 'van',
    'lorry', 'container_truck', 'prime_mover', 'tipper_truck', 'taxi',
]

# Label stems are cam{camera_id}_{YYYYMMDD}_{HHMMSS} — reconstruct source image path
def stem_to_image(stem: str) -> Path:
    """cam1001_20260309_141254 → raw/2026-03-09/1001/14-12-54.jpg"""
    _, cam_id, date_s, time_s = stem.split('_', 3) if stem.count('_') >= 3 else (None, None, None, None)
    # stem format: cam{id}_{YYYYMMDD}_{HHMMSS}  →  split on first _ gives 'cam{id}', '{YYYYMMDD}', '{HHMMSS}'
    parts = stem.split('_')          # ['cam1001', '20260309', '141254']
    cam_id = parts[0][3:]            # strip 'cam' prefix  → '1001'
    date_s = parts[1]                # '20260309'
    time_s = parts[2]                # '141254'
    date   = f'{date_s[:4]}-{date_s[4:6]}-{date_s[6:]}'   # '2026-03-09'
    time_  = f'{time_s[:2]}-{time_s[2:4]}-{time_s[4:]}'   # '14-12-54'
    return Path(RAW_DIR) / date / cam_id / f'{time_}.jpg'

# Quick sanity — verify a few label files resolve to real images
label_files = sorted(Path(LABELS_DIR).glob('*.txt'))
print(f'GDino label files found: {len(label_files)}')
if label_files:
    for lf in label_files[:3]:
        img = stem_to_image(lf.stem)
        print(f'  {lf.name}  →  {img}  ({"✓" if img.exists() else "✗ missing"})')

In [ ]:
import random
from collections import defaultdict

random.seed(42)

TRAIN_FRAC, VAL_FRAC = 0.70, 0.15   # remaining 0.15 → test

# ── 1. Group labels by camera ──────────────────────────────────────────────────
by_camera = defaultdict(list)
for lf in label_files:
    cam_id = lf.stem.split('_')[0]   # 'cam1001'
    by_camera[cam_id].append(lf)

print(f'Cameras with GDino labels: {len(by_camera)}')

# ── 2. Stratified 70/15/15 split so every camera appears in all three splits ──
splits = {'train': [], 'val': [], 'test': []}
for cam_id, files in by_camera.items():
    random.shuffle(files)
    n = len(files)
    n_train = max(1, round(n * TRAIN_FRAC))
    n_val   = max(1, round(n * VAL_FRAC))
    splits['train'].extend(files[:n_train])
    splits['val'].extend(files[n_train:n_train + n_val])
    splits['test'].extend(files[n_train + n_val:])

for s, items in splits.items():
    print(f'  {s}: {len(items)} files')

# ── 3. Copy images + labels into yolo_dataset/ ────────────────────────────────
stats = {'total': 0, 'missing_image': 0, 'copied': 0, 'empty_label': 0}

for split, label_list in splits.items():
    img_out = Path(OUTPUT_DIR) / 'images' / split
    lbl_out = Path(OUTPUT_DIR) / 'labels' / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for lf in label_list:
        stats['total'] += 1
        img_src = stem_to_image(lf.stem)

        if not img_src.exists():
            stats['missing_image'] += 1
            continue

        dest_img = img_out / (lf.stem + '.jpg')
        if not dest_img.exists():
            shutil.copy(str(img_src), str(dest_img))

        dest_lbl = lbl_out / lf.name
        shutil.copy(str(lf), str(dest_lbl))

        if lf.stat().st_size == 0:
            stats['empty_label'] += 1
        stats['copied'] += 1

    n_imgs = len(list(img_out.glob('*.jpg')))
    n_lbls = len(list(lbl_out.glob('*.txt')))
    print(f'{split}: {n_imgs} images, {n_lbls} labels')

print(f'\nStats: {stats}')
print(f'Label coverage: {stats["copied"] - stats["empty_label"]}/{stats["copied"]} '
      f'({100*(stats["copied"]-stats["empty_label"])/max(stats["copied"],1):.1f}% non-empty)')

In [ ]:
import yaml

data_yaml = {
    'path': OUTPUT_DIR,
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc': 10,
    'names': CLASS_NAMES,
}

yaml_path = Path(OUTPUT_DIR) / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f'✅ data.yaml written → {yaml_path}')
for split in ['train', 'val', 'test']:
    imgs = len(list((Path(OUTPUT_DIR) / 'images' / split).glob('*.jpg')))
    lbls = len(list((Path(OUTPUT_DIR) / 'labels' / split).glob('*.txt')))
    print(f'  {split}: {imgs} images, {lbls} labels')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

COLORS = [
    'red', 'blue', 'green', 'orange', 'purple',
    'cyan', 'magenta', 'brown', 'pink', 'gray',
]  # one per class: car motorcycle scooter bus van lorry container_truck prime_mover tipper_truck taxi

train_lbls = [p for p in (Path(OUTPUT_DIR) / 'labels' / 'train').glob('*.txt') if p.stat().st_size > 0]
if not train_lbls:
    print('No labeled images found')
else:
    lbl_path = random.choice(train_lbls)
    img_path = Path(OUTPUT_DIR) / 'images' / 'train' / (lbl_path.stem + '.jpg')

    if not img_path.exists():
        print(f'Image not found: {img_path}')
    else:
        img = Image.open(img_path)
        w, h = img.size

        fig, ax = plt.subplots(1, figsize=(10, 6))
        ax.imshow(img)
        for line in lbl_path.read_text().strip().split('\n'):
            if not line:
                continue
            cls, cx, cy, bw, bh = map(float, line.split())
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            color = COLORS[int(cls)]
            rect = patches.Rectangle((x1, y1), bw * w, bh * h,
                                      linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 5, CLASS_NAMES[int(cls)], color=color, fontsize=9, fontweight='bold')
        ax.set_title(img_path.name)
        plt.tight_layout()
        plt.show()
        print(f'Labels:\n{lbl_path.read_text()}')